# 3D Fluorescence Nuclei Segmentation

This notebook is a **generic template** for 3D fluorescence image segmentation and quantification of nuclei and cell-associated markers.

**Before running the workflow:**
- Replace the example file path, ROI, experiment name, and channel definitions in **Section 3** with your own data.
- Make sure the channel names in `stain_dict` match the metadata printed after loading your file.
- Start with the default settings and tune only the parameters that are necessary for your dataset.

---

# 1. Install Required Packages (run only once)
If you see errors about missing packages, run this cell.

In [ ]:
#[Cell 1]
from helpers.notebook_setup_helpers import install_required_packages

install_required_packages()

# 2. Import Required Libraries
This cell imports all necessary libraries for image processing and visualization.
If you see an ImportError, make sure to install the missing package using pip (see cell 1).

In [ ]:
#[Cell 2]
from helpers.notebook_setup_helpers import load_nuclei_notebook_setup
globals().update(load_nuclei_notebook_setup())

#### Helper Functions



The reusable helper functions for this notebook are stored in `helpers/notebook_helpers.py`.

This keeps the notebook much lighter and makes the workflow sections easier to follow.

If you update the helper file, rerun the next code cell to reload the helpers.


In [ ]:
#[Cell 3]
from importlib import reload

import helpers.notebook_helpers as nuclei_helpers

reload(nuclei_helpers)
from helpers.notebook_helpers import *

print("Helper functions loaded from helpers/notebook_helpers.py")

# 3. Inputs and Setup

Replace the example values in the following cells with information from **your own dataset** before running the analysis.

In this section, you will:
- Provide the image file path (`.nd2`, `.tif`, or similar)
- Define your marker/channel mapping
- Optionally crop a region of interest (ROI)
- Set approximate object sizes and processing options

> The `NUCLEI` entry in `stain_dict` is required, and the channel names should match the file metadata exactly.

### File input and ROI

In [ ]:
#[Cell 4]
# --- User input: file path and optional ROI ---
# Replace `input_file` with your own microscopy file.
# Supported examples include `.nd2` and multi-channel `.tif/.tiff` files.
input_file = 'path/to/your_image_file.nd2'

# Set to True to load a selected ROI from large datasets.
# Set to False to load the full stack directly.
big_image = True

# ROI format: [x0, x1, y0, y1, z0, z1]
# Use 0 for any limit you want to keep unchanged.
# Example: ROI = [100, 900, 150, 1200, 0, 40]
ROI = [0, 0, 0, 0, 0, 0]
ROI_print = ROI  # For display in processing parameters summary

In [ ]:
#[Cell 5]
meta = AICSImage(input_file)
if big_image:
    x0, x1, y0, y1, z0, z1 = ROI
    if x1==0:
        x1 = meta.shape[4]
    if y1==0:
        y1 = meta.shape[3]
    if z1==0:
        z1 = meta.shape[2]
    # Get lazy dask array in a known order, e.g. "TCZYX"
    lazy = meta.get_image_dask_data("ZYXC")
    sub = lazy[z0:z1, y0:y1, x0:x1, :]
    # Actually load this subset into memory
    img = sub.compute()
    ROI = [0, 0, 0, 0, 0, 0]
else:
    img = meta.get_image_data("XYZ", T=0) 

print(img.shape)

In [ ]:
#[Cell 6]
# Read voxel size metadata from the file.
# The channel names printed below should be used when editing `stain_dict`.
r_X = meta.physical_pixel_sizes.X  # um/px
r_Y = meta.physical_pixel_sizes.Y  # um/px
r_Z = meta.physical_pixel_sizes.Z  # um/px
print([r_X, r_Y, r_Z])

if big_image == False:
    imdata = meta.get_image_data()
    imtype = imdata.dtype
    bdepth = imtype.itemsize * 8
    print(imtype)

# Works for .nd2, .czi, .tif and other formats supported by AICSImage
file_meta = read_file_metadata(input_file, meta)
print("Date:", file_meta["date"])
print("Channels:", file_meta["channels"])


### Sample and channel metadata

In [ ]:
#[Cell 8]
# --- User input: biological and channel metadata ---
# Replace the example values below with those appropriate for your sample.

# Approximate object diameters used during segmentation.
nuclei_diameter = 10  # um, typical nucleus diameter in your sample
cell_diameter = 30  # um, typical whole-cell diameter if cytoplasm is estimated

# Relative expansion factors used when generating cytoplasm and PCM masks.
cyto_factor = int(np.round(cell_diameter/nuclei_diameter))
PCM_factor = int(cyto_factor * 1.1)
if PCM_factor == int(cyto_factor):
    PCM_factor += 1

# aggregate_grow_factor: factor by which cytoplasm labels are grown to detect aggregates.
#   - Controls how far each cytoplasm region is expanded when identifying cell aggregates.
#   - Default 2.0 applies moderate growth; increase for larger or more spread-out aggregates.
#   - Lower values (e.g. 1.5) give more conservative, tighter aggregate detection.
aggregate_grow_factor = 2.0

# Define the channels present in your dataset.
# Format:
# 'CONDITION_NAME': ['Marker label', 'channel name from file metadata', 'display color']
# Notes:
# - Keep the `NUCLEI` condition for the nuclear stain.
# - Replace the example channel names below so they match your file exactly.
stain_dict = {
    'NUCLEI': ['MARKER_NUCLEI', 'CHANNEL_NUCLEI', 'COLOR_NUCLEI'],
    'STAIN_1': ['MARKER_1', 'CHANNEL_1', 'COLOR_1'],
    'STAIN_2': ['MARKER_2', 'CHANNEL_2', 'COLOR_2'],
    'STAIN_3': ['MARKER_3', 'CHANNEL_3', 'COLOR_3'],
}

# Optional: list marker labels that should contribute to cytoplasm expansion
# if you do not have an explicit `CYTOPLASM` channel.
cyto_markers = ['MARKER_1', 'MARKER_2']  # replace with your own marker labels, or set to []

# Nucleus splitting by shrinking is configured in the helper module.
# Recommended start: "balanced"; use "aggressive" for clustered nuclei,
# or "conservative" if single nuclei are being over-split.
nuclei_split_config = get_nuclei_split_config(profile="balanced")

# Optional example overrides:
# nuclei_split_config["nuclei_seed_min_fraction"] = 0.05
# nuclei_split_config["nuclei_min_roundness"] = 0.40


### Image

In [ ]:
#[Cell 9]
# Optional scaling factors.
# Keep these values at 1.0 unless you intentionally want to apply an extra manual scale correction.
scale_factor = 1.0
zoom_factors = [1.0, 1.0, 1.0]  # [Z, Y, X]
zoom_factors = [x * scale_factor for x in zoom_factors]

### Setup

In [ ]:
#[Cell 10]
# --- User input: experiment-level processing options ---
# This prefix is used when saving setup files and output tables.
name_setup = 'your_experiment_name'

# Reuse previously saved napari contrast/gamma settings if the CSV file exists.
use_setup = True

# Set to True to use the StarDist model instead of watershed-based nuclei segmentation.
trig_stardist = False

# Set to True to compute multi-marker combinations in the quantification summary.
multilabel = True

# 4. Information

This section calculates derived image parameters (cell and nuclei dimensions, volumes), builds the staining table from your inputs, and opens an interactive Napari viewer for contrast and gamma setup.

In [ ]:
#[Cell 11]
im_final_stack, nuclei_radius, cell_radius, nuclei_volume, cell_volume = prepare_image_stack(
    img, meta, ROI, big_image, nuclei_diameter, cell_diameter
)


### Information about the staining

In [ ]:
#[Cell 12]
# Build the staining table from `stain_dict`.
# Channel names must match the metadata printed in the cell above.
stain_df = build_stain_dataframe(stain_dict, file_meta)

In [ ]:
#[Cell 13]
# Visualize each channel of the original image in a napari viewer.
viewer_0 = view_original_channels(im_final_stack, stain_df, napari, progress=tqdm)

### Acquisition processing setup

In [ ]:
#[Cell 14]
# Setup for acquisition and contrast/gamma settings
stain_df, stain_complete_df, original_stain_complete_df = prepare_stain_settings(
    im_final_stack['Original image'],
    stain_df=stain_df,
    name_setup=name_setup,
    use_setup=use_setup,
    settings=settings,
    napari_module=napari,
    progress=tqdm,
)

In [ ]:
#[Cell 15]
# Display stain settings DataFrame
stain_complete_df

# 5. Image Processing and Segmentation

This section normalizes the image, resamples to isotropic voxels, denoises, applies contrast and gamma adjustments, and then segments the structures of interest.

In [ ]:
#[Cell 16]
# Load and normalize image data for all channels
im_final_stack['Normalized image'] = normalize_image_channels(im_final_stack['Original image'])

# Plot histogram for each channel
hist_plot(im_final_stack['Normalized image'], stain_complete_df)


In [ ]:
#[Cell 17]
# Adapt resolution to isotropic
im_final_stack['Zoomed image'], r_zX, r_zY, r_zZ = resample_to_isotropic(
    im_final_stack['Normalized image'],
    zoom_factors,
    meta=meta
)

hist_plot(im_final_stack['Zoomed image'], stain_complete_df)


In [ ]:
#[Cell 18]
# Noise removal using median filter
im_final_stack['Denoised image'] = apply_median_denoise(im_final_stack['Zoomed image'])
hist_plot(im_final_stack['Denoised image'], stain_complete_df)


In [ ]:
#[Cell 19]
# Contrast and gamma adjustment for each channel
im_final_stack['Adjusted image'] = apply_contrast_gamma_per_channel(
    im_final_stack['Denoised image'],
    stain_complete_df
)
hist_plot(im_final_stack['Adjusted image'], stain_complete_df)


In [ ]:
#[Cell 20]
# --- User input ---
# sigma: standard deviation (in voxels) for the Gaussian smoothing kernel.
#   - Controls the trade-off between noise suppression and preservation of fine structure.
#   - Default 0.5 applies mild smoothing; increase to 1.0–2.0 for noisier acquisitions.
#   - Set to 0 to skip smoothing (not recommended for noisy images).
sigma = 0.5

# Gaussian filter for smoothing
im_final_stack['Filtered image'] = apply_gaussian_smoothing(
    im_final_stack['Adjusted image'],
    sigma
)
hist_plot(im_final_stack['Filtered image'], stain_complete_df)


In [ ]:
#[Cell 21]
# --- User input ---
# num_plateaus: number of clipping plateaus used during histogram equalization.
#   - Higher values redistribute intensity more aggressively, boosting contrast in dim regions.
#   - Default 2 is a good starting point; increase to 3–4 for low-contrast or dim channels.
#   - Setting to 1 approaches standard CLAHE-style single-plateau clipping.
num_plateaus = 2

# plateau_factor: controls the clipping height relative to the mean histogram bin count.
#   - Lower values apply stronger clipping (more aggressive equalization of bright spots).
#   - Higher values are gentler and preserve more of the original intensity distribution.
#   - Typical range: 0.5–1.0. Decrease toward 0.5 for heavily skewed histograms.
plateau_factor = 0.7

# Histogram equalization, supporting thresholding
im_final_stack['Equalized image'] = apply_histogram_equalization_per_channel(
    im_final_stack['Filtered image'],
    num_plateaus,
    plateau_factor
)
hist_plot(im_final_stack['Equalized image'], stain_complete_df)


In [ ]:
#[Cell 22]
# --- User input ---
# threshold_method: global algorithm used as one component (15%) of the combined threshold.
#   The Sauvola local threshold and statistical background component are always applied.
#   - 'otsu'  : maximises inter-class variance (default; robust for bimodal histograms).
#   - 'median': threshold at the median intensity of non-zero voxels
#               (good when the signal occupies a small fraction of the volume).
#   - 'huang' : fuzzy entropy method (more sensitive; useful for dim or diffuse signals).
threshold_method = 'otsu'

# Thresholding using global method + Sauvola local + statistical background + gain filtering
im_final_stack["Threshold image"] = apply_threshold_per_channel(
    im_final_stack["Equalized image"],
    stain_complete_df=stain_complete_df,
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    r_zxyz=(r_zX, r_zY, r_zZ),
    threshold_method=threshold_method,
    progress=tqdm,
)


In [ ]:
#[Cell 23]
# Export histograms for every processing stage (one sheet per channel per stage)
# and a Parameters sheet recording all stain and image processing settings.
output_path = Path(input_file).stem + '_histograms.xlsx'

processing_params = {
    'Input file':                     str(input_file),
    'ROI [x0,x1,y0,y1,z0,z1]':        str(ROI_print),
    'Big image mode':                 str(big_image),
    'Experiment name':                str(name_setup),
    'Nuclei diameter (um)':           str(nuclei_diameter),
    'Cell diameter (um)':             str(cell_diameter),
    'Cyto factor':                    str(cyto_factor),
    'PCM factor':                     str(PCM_factor),
    'Zoom factors [Z,Y,X]':           str(zoom_factors),
    'Scale factor':                   str(scale_factor),
    'Use StarDist':                   str(trig_stardist),
    'Multilabel':                     str(multilabel),
    'Cyto markers':                   str(cyto_markers),
    'Nuclei split config':            str(nuclei_split_config),
    'Sigma (Gaussian smoothing)':     str(sigma),
    'Num plateaus (histogram eq.)':   str(num_plateaus),
    'Plateau factor (histogram eq.)': str(plateau_factor),
    'Threshold method':               str(threshold_method),
    'Aggregate grow factor':          str(aggregate_grow_factor),
}

export_channel_histograms(
    im_final_stack,
    stain_complete_df,
    output_path,
    processing_params=processing_params,
    progress=tqdm,
)
print(f"Saved to: {output_path}")


## Segmentation

In [ ]:
#[Cell 24]
# Segmentation of nuclei using watershed or StarDist
im_segmentation_stack = segment_nuclei(
    im_final_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    nuclei_split_config=nuclei_split_config,
    r_zxyz=(r_zX, r_zY, r_zZ),
    nuclei_diameter=nuclei_diameter,
    trig_stardist=trig_stardist,
    progress=tqdm,
)


In [ ]:
#[Cell 25]
# Segmentation of cytoplasm
im_segmentation_stack, stain_complete_df = segment_cytoplasm(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    cyto_markers=cyto_markers,
    cyto_factor=cyto_factor,
    nuclei_diameter=nuclei_diameter,
    r_zxyz=(r_zX, r_zY, r_zZ),
    progress=tqdm,
)


In [ ]:
#[Cell 26]
# Segmentation of the pericellular matrix (PCM)
im_segmentation_stack = segment_pcm(
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    cyto_markers=cyto_markers,
    cyto_factor=cyto_factor,
    PCM_factor=PCM_factor,
)


In [ ]:
#[Cell 27]
# Assign segmented nuclei labels to other channels (cell assignment)
im_segmentation_stack = assign_channel_labels(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    progress=tqdm,
)


In [ ]:
#[Cell 28]
# Evaluate aggregates
if ('NUCLEI' in stain_df.index) | ('CYTOPLASM' in stain_df.index):
    im_out, num_aggregates = label(grow_labels(im_segmentation_stack['Cytoplasm'], aggregate_grow_factor) > 0)
    im_segmentation_stack['Aggregates'] = im_out * (im_segmentation_stack['Cytoplasm'] > 0)


In [ ]:
#[Cell 29]
# Visualize original, denoised, filtered, corrected, thresholded, assigned, and segmented images
viewer_0, viewer_1 = view_processing_results(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    r_xyz=(r_X, r_Y, r_Z),
    r_zxyz=(r_zX, r_zY, r_zZ),
    napari_module=napari,
    progress=tqdm,
)


# 6. Quantification and Analysis

This section quantifies nuclei and cell properties, computes statistics, and visualizes distributions. Results are exported for further analysis.

In [ ]:
#[Cell 30]
# Quantify marker overlap and intensity per segmented object, build DataFrames
filtered_img = im_final_stack['Filtered image']
r_xyz = (r_zX, r_zY, r_zZ)

labels_df, truncated_df = build_labels_df(
    im_segmentation_stack,
    filtered_img,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    r_xyz=r_xyz,
    zooms=zoom_factors,
    multilabel=multilabel,
    progress=tqdm,
)

labels_df


In [ ]:
#[Cell 31]
# Print summary statistics for nuclei and cell populations
print_population_summary(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    progress=tqdm,
)


In [ ]:
#[Cell 32]
# Build the full quantification table and convert it to a DataFrame for exports and 3D outputs
r_xyz = (r_X, r_Y, r_Z)

labels_full_df = build_full_labels_df(
    im_segmentation_stack,
    im_final_stack,
    filtered_img,
    stain_complete_df=stain_complete_df,
    r_xyz=r_xyz,
    zooms=zoom_factors,
    progress=tqdm,
)


### HISTOGRAMS

In [ ]:
#[Cell 33]
# Collect histogram data, plot KDE distributions, and generate the per-nucleus PDF report
hist_data, intensity_ranges, fig, axes, x_grid = build_histogram_report(
    im_segmentation_stack,
    im_final_stack,
    filtered_img,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
    progress=tqdm,
)


In [ ]:
#[Cell 34]
# --- User input ---
# nuc_3D_export: set to True to export a cubic sub-volume centred on a chosen nucleus.
#   - When True, you will be prompted to enter the nucleus label number (integer).
#   - The sub-volume is saved as a VTK file in the working directory for 3D visualisation.
#   - Set to False (default) to skip this export step.
nuc_3D_export = False

# Export a single nucleus sub-volume as a VTK file (optional)
if nuc_3D_export:
    nuc_label = int(input("Enter nucleus label number to export: "))
    export_nucleus_vtk_crop(
        nuc_label=nuc_label,
        im_segmentation_stack=im_segmentation_stack,
        im_final_stack=im_final_stack,
        stain_df=stain_df,
        input_file=input_file,
    )


## Evaluate cell distribution in the space

In [ ]:
#[Cell 35]
# Plot spatial distribution of nuclei and cells
im_in = im_final_stack['Filtered image']
plot_spatial_distributions(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    im_in=im_in,
    r_X=r_X,
    r_Y=r_Y,
    r_Z=r_Z,
    zoom_factors=zoom_factors,
    progress=tqdm,
)


## Evaluate cell size distribution

In [ ]:
#[Cell 36]
# Plot size distribution of nuclei and cells
plot_size_distributions(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    progress=tqdm,
)


## CREATE .VTK VOLUME

In [ ]:
#[Cell 37]
# Build labelled VTK volume meshes for nuclei, cytoplasm and PCM
build_vtk_volumes(
    im_segmentation_stack,
    labels_full_df=labels_full_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
    r_xyz=(r_X, r_Y, r_Z),
    zoom_factors=zoom_factors,
    progress=tqdm,
)


## and .STL for markers

In [ ]:
#[Cell 38]
# Export per-marker binary volumes as STL mesh files
export_marker_stl(
    im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
    progress=tqdm,
)


### Create a complete report XSL

In [ ]:
#[Cell 39]
# Export quantification results to Excel file
output_path = Path(input_file).stem + '_segmentation.xlsx'
export_quantification_to_excel(
    output_path,
    original_stain_complete_df,
    labels_full_df,
    progress=tqdm,
)
print(f"Saved to: {output_path}")


# CREATE .inp FOR FINITE ELEMENT ANALYSIS

In [ ]:
#[Cell 40]
# Tetrahedralize nuclei surface, assign elements to nucleus labels, write Abaqus .inp
export_fea_mesh(
    im_segmentation_stack,
    input_file=input_file,
    progress=tqdm,
)